In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.5 Similarity, Schur, Jordan, and Non-normality

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Spectral Theory",
    number="3.5",
    title="Similarity, Schur, Jordan, and Non-normality",
    blurb="The Jordan form is exact, beautiful, and uncomputable; the Schur "
    "form is computable and always exists; and in between sits the honest "
    "answer to why a matrix whose every eigenvalue says decay can amplify a "
    "vector sixfold first.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

[§3.4](hermitian-unitary-normal.ipynb) found the exact condition for an
orthonormal eigenbasis — normality — and measured that its Schur form is
diagonal while a non-normal matrix's is not. This notebook is about what that
off-diagonal part *costs*, and the answer is: most of what one expects
eigenvalues to tell you.

Three decompositions, in decreasing order of beauty and increasing order of
usefulness. **Jordan** is the complete classification: every matrix is similar
to a direct sum of Jordan blocks, and that form settles every question about
similarity. It is also uncomputable in floating point, for a reason this
notebook makes quantitative — perturb a $6\times6$ Jordan block by $10^{-14}$
and its eigenvalues do not move by $10^{-14}$, they scatter onto a circle of
radius $\delta^{1/6} = 5\times10^{-3}$, eleven orders larger. **Schur** exists
for every matrix, is computed by a stable algorithm, and gives a unitary $Q$;
what it gives up is a diagonal middle. **Diagonalization** is what one wants
and cannot always have.

Then the consequences. **Bauer–Fike** replaces the perfect conditioning of
[§3.2](spectral-theorem.ipynb): eigenvalues move by at most
$\operatorname{cond}(X)\|E\|$ rather than $\|E\|$, and $\operatorname{cond}(X)$
can be $10^{6}$. **Transient growth** is the practical sting: a matrix with
spectral radius $0.8$ — every eigenvalue comfortably inside the unit disc, so
$A^k \to 0$ guaranteed — nevertheless has $\|A^4\| = 6.17$. Anything driven by
that matrix grows sixfold before it decays, and no eigenvalue predicts it.
**Pseudospectra** are the picture that does.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Trefethen and Embree's *Spectra and Pseudospectra*
> {cite}`trefethen2005spectra` is the book for the second half and is a
> pleasure to read; Golub and Van Loan {cite}`golub2013` §7.1 for Schur and
> Jordan; Horn and Johnson {cite}`horn2013` Chapter 3; Higham
> {cite}`higham2002` on why the Jordan form is not computed.

## Theory in brief

### Similarity

$A$ and $B$ are **similar** if $B = X^{-1}AX$ for some invertible $X$: the same
linear map written in two bases, exactly as
[§1.6](../01-matrices/linear-maps-change-of-basis.ipynb) described. Similarity
preserves the characteristic polynomial, hence the eigenvalues with their
multiplicities, hence the trace and determinant. It preserves *nothing* about
lengths: norms, singular values, symmetry and orthogonality of eigenvectors are
all free to change, and that gap is the subject of this notebook.

### Three decompositions

**Jordan.** Every $A \in \mathbb{C}^{n\times n}$ is similar to a block diagonal

```{math}
:label: eq-nn-jordan
A = PJP^{-1},
\qquad J = \operatorname{diag}\bigl(J_{n_1}(\lambda_1), \dots, J_{n_k}(\lambda_k)\bigr),
```

where $J_m(\lambda)$ has $\lambda$ on the diagonal and 1 on the superdiagonal.
This is a complete invariant: two matrices are similar exactly when their
Jordan forms agree up to block order.

**Schur.** Every $A$ has

```{math}
:label: eq-nn-schur
A = QTQ^{*},
\qquad Q \text{ unitary},\; T \text{ upper triangular},
```

with the eigenvalues on $\operatorname{diag}T$. Unlike {eq}`eq-nn-jordan` this
is computed by a backward-stable algorithm, and it is what `scipy.linalg.schur`
returns.

### Why Jordan cannot be computed

The Jordan structure is **discontinuous** in $A$. Perturb the $n\times n$
nilpotent block $J_n(0)$ by putting $\delta$ in its bottom-left corner. The
characteristic polynomial becomes $\lambda^n = \delta$, so the eigenvalues are

```{math}
:label: eq-nn-dissolution
\lambda_j = \delta^{1/n}\,e^{2\pi ij/n}, \qquad j = 0, \dots, n-1 :
```

$n$ distinct eigenvalues on a circle of radius $\delta^{1/n}$. The single
eigenvalue of multiplicity $n$ has dissolved, and the displacement is
$\delta^{1/n}$, not $\delta$. At $n = 6$ and $\delta = 10^{-14}$ that is
$5\times10^{-3}$ — a perturbation at the level of rounding error moving the
answer into the third decimal place. Any matrix is within $\varepsilon$ of a
diagonalizable one, so "is this matrix defective?" is not a question floating
point can answer.

### Bauer–Fike

If $A = X\Lambda X^{-1}$ is diagonalizable, then every eigenvalue $\mu$ of
$A + E$ satisfies

```{math}
:label: eq-nn-bauer-fike
\min_i |\mu - \lambda_i| \;\le\; \operatorname{cond}_2(X)\,\|E\|_2 .
```

The amplifier is the conditioning of the *eigenvector* matrix. For a normal
matrix $X$ is unitary and $\operatorname{cond}(X) = 1$, recovering
[§3.2](spectral-theorem.ipynb)'s Weyl bound exactly; for a non-normal matrix it
can be arbitrarily large.

### Transient growth, and the Kreiss lower bound

The spectral radius controls the *asymptotic* behaviour: $A^k \to 0$ if and
only if $\rho(A) < 1$. It says nothing about the way there. A matrix can
satisfy $\rho(A) < 1$ and still have $\|A^k\| \gg 1$ for a range of $k$ before
the decay sets in, and for a *normal* matrix this is impossible, since
$\|A^k\|_2 = \rho(A)^k$ exactly.

### Pseudospectra

For $\epsilon > 0$ the **$\epsilon$-pseudospectrum** has three equivalent
definitions:

```{math}
:label: eq-nn-pseudospectrum
\Lambda_\epsilon(A)
 = \bigl\{z : \|(zI - A)^{-1}\|_2 > \epsilon^{-1}\bigr\}
 = \bigl\{z : \sigma_{\min}(zI - A) < \epsilon\bigr\}
 = \bigl\{z : z \in \Lambda(A + E) \text{ for some } \|E\|_2 < \epsilon\bigr\} .
```

The second is how it is computed — one SVD per grid point — and the third is
what it means: the set of numbers that are eigenvalues of something within
$\epsilon$ of $A$. For a normal matrix $\Lambda_\epsilon$ is exactly the union
of $\epsilon$-discs around the eigenvalues, and nothing more; for a non-normal
matrix it bulges far beyond them, and how far is what predicts the transient.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.linalg import schur

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps
np.set_printoptions(precision=5, suppress=True, linewidth=120)


def jordan_block(n, lam=0.0):
    """The n-by-n Jordan block: lam on the diagonal, 1 on the superdiagonal."""
    return np.diag(np.full(n, float(lam))) + np.diag(np.ones(n - 1), 1)


def resolvent_norm(z, A):
    """||(zI - A)^-1||_2, computed as 1/sigma_min(zI - A) (Eq. 6).

    Going through the smallest singular value rather than forming the inverse is
    both cheaper and defined everywhere: at an eigenvalue sigma_min is zero and
    the resolvent norm is correctly infinite, where an explicit inverse would
    raise.
    """
    s = np.linalg.svd(z * np.eye(A.shape[0]) - A, compute_uv=False)
    smin = s.min()
    return np.inf if smin == 0.0 else 1.0 / float(smin)


# The transient-growth matrix: spectral radius 0.8, so A^k -> 0 guaranteed.
A_TRANS = np.array([[0.8, 3.0],
                    [0.0, 0.8]])
# A normal matrix with exactly the same spectrum, for comparison.
A_NORMAL = np.diag([0.8, 0.8])

## Exercise 1 — What similarity keeps, and what it throws away

$B = X^{-1}AX$ is the same map in a different basis. The characteristic
polynomial is unchanged, since $\det(X^{-1}AX - \lambda I) =
\det(X^{-1}(A - \lambda I)X) = \det(A - \lambda I)$, so the eigenvalues,
trace and determinant all survive. Almost nothing else does — and the
quantities that do not survive are exactly the ones that decide whether a
computation is well behaved.

**Part a)** Build $A = \left[\begin{smallmatrix}4&1\\2&3\end{smallmatrix}\right]$,
the shear $X = \left[\begin{smallmatrix}1&5\\0&1\end{smallmatrix}\right]$
(whose condition number is 27, so it distorts lengths substantially), and
$B = X^{-1}AX$ with `np.linalg.solve(X, A @ X)`. Report both matrices. A
*random* $X$ would not do here: a random $2\times2$ is often close to
orthogonal, and then the norm barely moves and the exercise's point is lost.

**Part b)** Confirm what is preserved, to $10^{-12}$: the eigenvalues (sorted),
the trace, and the determinant. The eigenvalues are 2 and 5.

**Part c)** Confirm what is *not*. Report $\|A\|_2$ against $\|B\|_2$ and the
singular values of each; they differ substantially. Similarity is not an
isometry, and a matrix norm is not a similarity invariant.

**Part d)** Confirm that symmetry is not preserved either. Take the symmetric
$S = \left[\begin{smallmatrix}3&1\\1&2\end{smallmatrix}\right]$ and form
$X^{-1}SX$ for the same $X$; confirm the result is *not* symmetric (report
$\|M - M^{\top}\|_{\max}$, which is order 1) while its eigenvalues are
unchanged to $10^{-12}$. This is why [§3.2](spectral-theorem.ipynb) insisted on
*orthogonal* similarity: $Q^{\top}SQ$ preserves symmetry and a general
$X^{-1}SX$ does not.

**Part e)** Confirm the one similarity that keeps everything. For a random
orthogonal $Q$ from `np.linalg.qr(rng.standard_normal((2, 2)))`, form
$Q^{\top}SQ$ and confirm it is symmetric to $10^{-14}$, has the same singular
values to $10^{-13}$, and the same 2-norm. Orthogonal similarity is a rotation
of the coordinate system; a general similarity is a shear, and a shear can
distort every length in sight.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The point of this exercise is the contrast, so both halves are checked: the
invariants to $10^{-12}$ and the *non*-invariants as quantities that visibly
moved. A check that only confirmed the eigenvalues survive would leave the
impression that similarity is harmless.

In [ ]:
validate.close(
    lam_B, lam_A,
    "similarity preserves the eigenvalues (2 and 5)",
    rtol=0.0, atol=1e-12,
)
validate.check(
    abs(np.trace(A_sim) - np.trace(B_sim)) < 1e-12
    and abs(np.linalg.det(A_sim) - np.linalg.det(B_sim)) < 1e-12,
    "and therefore the trace and determinant",
    f"trace {np.trace(A_sim):.6f} vs {np.trace(B_sim):.6f}, det "
    f"{np.linalg.det(A_sim):.6f} vs {np.linalg.det(B_sim):.6f}",
)
validate.check(
    abs(sv_B[0] / sv_A[0] - 1.0) > 0.5,
    "but NOT the norm: similarity is not an isometry",
    f"||A||_2 = {sv_A[0]:.4f} against ||B||_2 = {sv_B[0]:.4f}, a ratio of "
    f"{sv_B[0]/sv_A[0]:.4f}. Eigenvalues are similarity invariants; singular "
    "values are not",
)
validate.check(
    asym > 0.1 and np.abs(lam_M - lam_S).max() < 1e-12,
    "and not symmetry: a general similarity destroys it while keeping the spectrum",
    f"||M - M^T|| = {asym:.4f} with eigenvalues unchanged to "
    f"{np.abs(lam_M-lam_S).max():.1e}. This is why 3.2 required ORTHOGONAL "
    "similarity",
)
validate.check(
    sym_orth < 1e-14
    and np.abs(np.sort(sv_orth) - np.sort(sv_S)).max() < 1e-13,
    "an orthogonal similarity keeps symmetry, singular values and norm",
    f"||M - M^T|| = {sym_orth:.1e}, singular values agree to "
    f"{np.abs(np.sort(sv_orth)-np.sort(sv_S)).max():.1e}: a rotation of the "
    "coordinates rather than a shear",
)

## Exercise 2 — Schur: the decomposition that always exists

{eq}`eq-nn-schur` is the workhorse. It asks nothing of $A$, it uses a unitary
$Q$ so nothing is amplified, and it is what LAPACK computes on the way to the
eigenvalues. The price is that $T$ is triangular rather than diagonal, and
[§3.4](hermitian-unitary-normal.ipynb) established that the off-diagonal part
vanishes exactly for normal matrices.

The test matrices are the $8\times8$ Grcar matrix `la.grcar(8)` — a standard
non-normal example, Toeplitz with ones on the diagonal and three
superdiagonals and $-1$ below — and a random $6\times6$ real matrix.

**Part a)** For both matrices compute `T, Q = schur(A, output="complex")` and
confirm $A = QTQ^{*}$ to $10^{-12}$, that $Q^{*}Q = I$ to $10^{-13}$, and that
$T$ is upper triangular exactly (`np.tril(T, -1)` identically zero).

**Part b)** Confirm the diagonal of $T$ is the spectrum. Compare
`np.diag(T)` against `np.linalg.eigvals(A)` **as sorted sets**, using a
lexicographic sort on (real, imaginary) — not `np.sort_complex`, which
[§3.4](hermitian-unitary-normal.ipynb) showed orders on the real part alone.
Agreement should be $10^{-10}$.

**Part c)** Quantify the non-normality. Report
$\|T - \operatorname{diag}T\|_F$ for both matrices, and the **departure from
normality** $\Delta(A) = \sqrt{\|A\|_F^2 - \sum_i|\lambda_i|^2}$, which is zero
exactly for normal matrices. Confirm the two agree to $10^{-10}$: the
off-diagonal Frobenius mass of the Schur form *is* the departure from
normality, which makes {eq}`eq-nn-schur` a way of measuring it.

**Part d)** Confirm the real Schur form is different. Call `schur(A)` without
`output="complex"` on the random real matrix and confirm the result is real,
but only *quasi*-triangular: it has $2\times2$ blocks on the diagonal wherever
a complex-conjugate eigenvalue pair occurs. Report how many such blocks there
are, by counting nonzero entries on the first subdiagonal. Real arithmetic
cannot produce a triangular form with complex eigenvalues on the diagonal, and
this is the compromise.

**Part e)** Confirm the practical consequence: the Schur form gives the
eigenvalues *without* ever forming an eigenvector matrix, so nothing in it can
be badly conditioned. Report $\operatorname{cond}(Q)$ for both matrices — it is
1 — against $\operatorname{cond}(X)$ from `np.linalg.eig`, which for the Grcar
matrix is larger. Exercise 4 makes that number matter.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The identity $\|T - \operatorname{diag}T\|_F = \Delta(A)$ is the check worth
having: it connects the Schur form's shape to a quantity defined without
reference to any decomposition, so the two could disagree and do not.

In [ ]:
validate.check(
    max(r[1] for r in schur_rows) < 1e-12 and max(r[2] for r in schur_rows) < 1e-13,
    "A = Q T Q* holds for both matrices with Q unitary (Eq. 2)",
    f"worst reconstruction {max(r[1] for r in schur_rows):.2e}, worst |Q*Q - I| "
    f"{max(r[2] for r in schur_rows):.2e}",
)
validate.check(
    all(r[3] == 0.0 for r in schur_rows),
    "with T exactly upper triangular",
    "the strictly lower triangle is identically zero in both cases",
)
validate.check(
    max(r[4] for r in schur_rows) < 1e-10,
    "and diag(T) is the spectrum, compared as a lexicographically sorted set",
    f"worst gap {max(r[4] for r in schur_rows):.2e}. np.sort_complex would not "
    "do here: it orders on the real part alone and leaves conjugate pairs in "
    "input order",
)
validate.check(
    all(abs(r[5] - r[6]) < 1e-10 for r in schur_rows),
    "the off-diagonal Frobenius mass of T IS the departure from normality",
    f"Grcar: {schur_rows[0][5]:.6f} against {schur_rows[0][6]:.6f}; random: "
    f"{schur_rows[1][5]:.6f} against {schur_rows[1][6]:.6f}. The second is "
    "defined without reference to any decomposition, so they could disagree",
)
validate.check(
    all(abs(r[7] - 1.0) < 1e-12 for r in schur_rows) and n_blocks >= 1,
    "cond(Q) = 1 always, and the real Schur form is quasi-triangular",
    f"cond(Q) = 1 to 1e-12 for both; the real form has {n_blocks} nonzero "
    "subdiagonal entries, one per complex-conjugate pair — real arithmetic "
    "cannot put a complex eigenvalue on a diagonal",
)

## Exercise 3 — The Jordan form dissolves, and the rate is exactly $\delta^{1/n}$

{eq}`eq-nn-jordan` is the complete answer to the similarity classification
problem and it cannot be computed. This exercise shows why, and the failure is
not a matter of degree: it is a discontinuity with an exactly known rate.

Take the nilpotent block $J_n(0)$ and put $\delta$ in the bottom-left corner.
Expanding the determinant gives $\det(J_n(0) - \lambda I + \delta e_ne_1^{\top})
= (-\lambda)^n + (-1)^{n+1}\delta$, so the characteristic equation is
$\lambda^n = \delta$ and the eigenvalues are exactly
{eq}`eq-nn-dissolution`: $n$ of them, equally spaced on a circle of radius
$\delta^{1/n}$.

**Part a)** Confirm the exact Jordan form is available symbolically. For
$A = \left[\begin{smallmatrix}2&1&0\\0&2&1\\0&0&2\end{smallmatrix}\right]$ use
`sp.Matrix(...).jordan_form()` to get $(P, J)$ and confirm
$PJP^{-1} = A$ **exactly** with `sp.simplify(...) == sp.zeros(3, 3)`. Report
$J$: one $3\times3$ block with eigenvalue 2.

**Part b)** Now perturb numerically. For $n = 4, 6, 8$ and
$\delta = 10^{-10}, 10^{-14}$, build `jordan_block(n)`, add $\delta$ at
`[n-1, 0]`, and report $|\lambda|$ for the resulting eigenvalues alongside the
prediction $\delta^{1/n}$.

**Part c)** Confirm {eq}`eq-nn-dissolution` to a relative $10^{-6}$: every
eigenvalue has modulus $\delta^{1/n}$, and they are equally spaced in angle.
Check the spacing by sorting the arguments and confirming consecutive
differences equal $2\pi/n$ to $10^{-6}$.

**Part d)** State the amplification. At $n = 6$, $\delta = 10^{-14}$ the
eigenvalues move to $4.6\times10^{-3}$ — a displacement $4.6\times10^{11}$
times the perturbation. Confirm the ratio $\delta^{1/n}/\delta$ exceeds
$10^{10}$ for that case. A perturbation at the level of rounding error has
moved the answer into the third decimal place.

**Part e)** Draw the conclusion the right way round. It is not that the
algorithm is bad: *any* matrix is within $\varepsilon$ of a diagonalizable one,
because the diagonalizable matrices are dense. Confirm that directly by taking
the $6\times6$ Jordan block, adding a random perturbation of norm $10^{-13}$,
and checking the result has 6 distinct eigenvalues — hence is diagonalizable —
over 20 trials. "Is this matrix defective?" is not a question `float64` can
answer, and that is why `scipy` offers `schur` and not `jordan`.

In [ ]:
# (solution hidden on the public site)


### Validation 3

The dissolution rate is checked against an exact closed form rather than an
order-of-magnitude estimate, and the *angles* are checked too, because
"$n$ eigenvalues on a circle" is a stronger and more falsifiable claim than
"the eigenvalues moved a lot".

In [ ]:
validate.check(
    jordan_exact,
    "SymPy's Jordan form is exact over the rationals: P J P^-1 = A (Eq. 1)",
    "one 3x3 block with eigenvalue 2 — algebraic multiplicity 3, geometric 1",
)
validate.check(
    max(mod_rel) < 1e-6,
    "the perturbed eigenvalues have modulus exactly delta^(1/n) (Eq. 3)",
    f"worst relative deviation {max(mod_rel):.2e} over n = 4, 6, 8 and "
    "delta = 1e-10, 1e-14: the characteristic equation is lambda^n = delta, so "
    "this is a closed form and not an estimate",
)
validate.check(
    max(angle_gaps) < 1e-6,
    "and they are equally spaced in angle, 2 pi / n apart (Eq. 3)",
    f"worst angular deviation {max(angle_gaps):.2e}: n distinct eigenvalues on a "
    "circle, where there had been one of multiplicity n",
)
validate.check(
    amp > 1e10,
    "so a 1e-14 perturbation moves the answer by 4.6e-3, an amplification of 5e11",
    f"delta^(1/6)/delta = {amp:.2e}. Nothing is wrong with the algorithm: the "
    "Jordan STRUCTURE is discontinuous in A",
)
validate.check(
    distinct == 20,
    "and every one of 20 tiny random perturbations made J_6(0) diagonalizable",
    "the diagonalizable matrices are dense in C^(n x n), so 'is this matrix "
    "defective?' is not a question float64 can answer — which is why scipy "
    "offers schur and not jordan",
)

## Exercise 4 — Bauer–Fike: the conditioning of the eigenvector matrix

[§3.2](spectral-theorem.ipynb) established that symmetric eigenvalues are
perfectly conditioned: a perturbation of norm $\|E\|$ moves no eigenvalue
further than $\|E\|$. {eq}`eq-nn-bauer-fike` is the general statement, and the
difference is a factor of $\operatorname{cond}(X)$ — which for a normal matrix
is 1 and otherwise is not.

The test matrix is $A = X\Lambda X^{-1}$ with
$\Lambda = \operatorname{diag}(1, 2, 3)$ and the deliberately ill-conditioned

$$
X = \begin{bmatrix} 1 & 1 & 1\\ 0 & 10^{-3} & 2\times10^{-3}\\
                    0 & 0 & 10^{-6}\end{bmatrix},
$$

whose condition number is $4.2\times10^{6}$: the eigenvectors are nearly
parallel, which is what makes the eigenvalues sensitive.

**Part a)** Build $A$ and confirm its eigenvalues are $1, 2, 3$ to $10^{-8}$,
and report $\operatorname{cond}(X) = 4.2\times10^{6}$.

**Part b)** Apply 100 random perturbations $E$ of norm about $10^{-6}$, built
as `1e-6 * rng.standard_normal((3, 3))`. For each, compute the eigenvalues of
$A + E$ and the largest distance from any of them to the nearest of
$\{1, 2, 3\}$. Confirm {eq}`eq-nn-bauer-fike` holds every time: that distance
never exceeds $\operatorname{cond}(X)\|E\|_2$.

**Part c)** Report the worst ratio $\min_i|\mu - \lambda_i| \big/
(\operatorname{cond}(X)\|E\|_2)$ over the 100 trials. It comes out about
$0.15$, so the bound holds with room to spare — Bauer–Fike is a worst case, and
a worst case attained only by adversarial perturbations. Report also the actual
eigenvalue movement in absolute terms.

**Part d)** Contrast with the symmetric case. Take a symmetric matrix with the
same spectrum, $S = Q\operatorname{diag}(1,2,3)Q^{\top}$ for a random
orthogonal $Q$, apply the *same* 100 perturbations symmetrised as
$(E + E^{\top})/2$, and confirm the movement never exceeds $\|E\|_2$ — Weyl's
bound, with $\operatorname{cond}(X) = 1$. Report the ratio of the two worst
movements: the non-normal matrix's eigenvalues move far further under
perturbations of the same size.

**Part e)** Confirm the mechanism is the eigenvector conditioning and not
anything else, by checking that $\operatorname{cond}$ of the eigenvector matrix
returned by `np.linalg.eig(A)` is itself around $10^{6}$, while for $S$ it is
1 to $10^{-12}$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

Bauer–Fike is checked as an inequality over 100 trials, which is the only
honest way to test a bound, and the *worst ratio* is reported so the reader can
see it is not attained. The symmetric comparison is run with the same
perturbations so the difference cannot be attributed to the draw.

In [ ]:
validate.close(
    lam_bf, targets,
    "the test matrix has eigenvalues 1, 2, 3 despite cond(X) = 4e6",
    rtol=0.0, atol=1e-8,
)
validate.check(
    max(bf_ratios) <= 1.0,
    "Bauer-Fike holds for all 100 perturbations (Eq. 4)",
    f"worst ratio |dlambda| / (cond(X) ||E||) = {max(bf_ratios):.6f}, so the "
    "bound holds with room: it is a worst case, attained only by adversarial E",
)
validate.check(
    max(weyl_ratios) <= 1.0,
    "and Weyl holds for the symmetric matrix with the same spectrum",
    f"worst ratio |dlambda| / ||E|| = {max(weyl_ratios):.6f}: cond(X) = 1 there, "
    "so Bauer-Fike degenerates to Weyl exactly",
)
validate.check(
    max(bf_moves) > 100 * max(weyl_moves),
    "but the non-normal eigenvalues move orders further for the same ||E||",
    f"{max(bf_moves):.3e} against {max(weyl_moves):.3e}, a factor of "
    f"{max(bf_moves)/max(weyl_moves):.0f}, under the SAME 100 perturbations",
)
validate.check(
    cond_eig_A > 1e5 and abs(cond_eig_S - 1.0) < 1e-12,
    "and the amplifier is exactly the eigenvector matrix's conditioning",
    f"cond(X) = {cond_eig_A:.2e} for A against {cond_eig_S:.6f} for S",
)

## Exercise 5 — Spectral radius below 1, and the norm goes up anyway

This is the practical sting of non-normality. The spectral radius answers one
question exactly — does $A^k \to 0$? — and people routinely read it as
answering a different one: does $\|A^k\|$ decrease? For a normal matrix those
are the same question, since $\|A^k\|_2 = \rho(A)^k$. For a non-normal matrix
they are not, and the gap can be large.

The matrix is $A = \left[\begin{smallmatrix}0.8 & 3\\ 0 & 0.8\end{smallmatrix}\right]$,
available as `A_TRANS`. Both eigenvalues are $0.8$, so $\rho = 0.8 < 1$ and
$A^k \to 0$ is guaranteed. The comparison is
$\operatorname{diag}(0.8, 0.8)$, available as `A_NORMAL`: same spectrum,
normal.

**Part a)** Confirm $\rho(A_{\text{TRANS}}) = 0.8$ exactly and that
$A_{\text{TRANS}}$ is not normal, reporting $\|AA^{*} - A^{*}A\|$.

**Part b)** Compute $\|A^k\|_2$ for $k = 0, \dots, 59$ using
`np.linalg.matrix_power` and report the maximum and where it occurs. It reaches
$6.17$ at $k = 4$: the matrix amplifies by more than a factor of six before it
begins to decay.

**Part c)** Confirm the decay does eventually happen, as the spectral radius
promises: report $\|A^{59}\|_2$, which is $4\times10^{-4}$. The asymptotic
statement was true all along; it was simply not the statement one wanted.

**Part d)** Confirm the normal matrix cannot do this. For `A_NORMAL`, confirm
$\|A^k\|_2 = 0.8^k$ to $10^{-14}$ for every $k$, so the maximum is $1$ at
$k = 0$ and the sequence never rises. This is the exact sense in which
non-normality is *necessary* for a transient.

**Part e)** Confirm the transient scales with the off-diagonal entry. For
$a = 0.9, 1.5, 3$ in $\left[\begin{smallmatrix}0.8&a\\0&0.8\end{smallmatrix}\right]$,
report the peak $\|A^k\|$; it grows roughly linearly in $a$ while the spectrum
is *identical* in all three cases. Confirm the peak values are increasing and
that all three matrices have exactly the same eigenvalues, to $10^{-15}$.
Nothing about the spectrum distinguishes them, and the behaviour differs by a
factor of three.

**Part f)** Plot $\|A^k\|_2$ against $k$ on a log axis for the three values of
$a$ and for the normal matrix, with $0.8^k$ drawn as the asymptotic reference.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

The normal comparison is what turns this from an anecdote into a theorem: the
transient is not merely *possible* for a non-normal matrix, it is *impossible*
for a normal one, and $\|A^k\| = \rho^k$ to $10^{-16}$ is the exact statement
of that.

In [ ]:
validate.check(
    abs(rho_trans - 0.8) < 1e-15 and comm_trans > 0.5,
    "A_TRANS has spectral radius 0.8 and is not normal",
    f"rho = {rho_trans:.6f} < 1, so A^k -> 0 is guaranteed; commutator "
    f"{comm_trans:.1f}",
)
validate.check(
    norms_trans.max() > 6.0 and norms_trans[-1] < 1e-3,
    "yet ||A^k|| reaches 6.17 before decaying to 4e-4",
    f"peak {norms_trans.max():.4f} at k = {ks[norms_trans.argmax()]}, and "
    f"||A^59|| = {norms_trans[-1]:.3e}. The asymptotic claim was true; it was "
    "not the claim one wanted",
)
validate.close(
    norms_norm, 0.8 ** ks,
    "for the normal matrix ||A^k||_2 = rho^k exactly, so it can never rise",
    rtol=0.0, atol=1e-14,
)
validate.check(
    peaks == sorted(peaks) and spec_same < 1e-15,
    "the transient grows with the off-diagonal entry at IDENTICAL spectrum",
    f"peaks {[f'{p:.3f}' for p in peaks]} for a = 0.9, 1.5, 3.0, whose spectra "
    f"agree to {spec_same:.1e}: a {peaks[-1]/peaks[0]:.1f}-fold difference in "
    "behaviour that no eigenvalue sees",
)

## Exercise 6 — Pseudospectra: the picture that does predict it

Eigenvalues failed to predict the transient because they are the wrong object
for a non-normal matrix. {eq}`eq-nn-pseudospectrum` gives the right one: the
set of $z$ that are eigenvalues of *some* matrix within $\epsilon$ of $A$.
For a normal matrix that set is exactly the union of $\epsilon$-discs about the
eigenvalues; for a non-normal one it bulges, and how far it bulges is what the
behaviour follows.

The three definitions in {eq}`eq-nn-pseudospectrum` are equivalent, and the
middle one — $\sigma_{\min}(zI - A) < \epsilon$ — is how it is computed: one
SVD per grid point, which is why pseudospectra were impractical until they were
not.

**Part a)** Confirm the resolvent and singular-value definitions agree. For the
$8\times8$ Grcar matrix and 200 random points $z$ in the box
$[-2, 4]\times[-3, 3]$, confirm
$\|(zI - A)^{-1}\|_2 = 1/\sigma_{\min}(zI - A)$ to a relative $10^{-10}$,
computing the left side with `np.linalg.inv` and the right with
`np.linalg.svd(..., compute_uv=False)`.

**Part b)** Confirm the spectrum is inside every pseudospectrum: for each
eigenvalue $\lambda$ of the Grcar matrix, confirm
$\sigma_{\min}(\lambda I - A) < 10^{-12}$, so the resolvent norm exceeds
$10^{12}$ and $\lambda \in \Lambda_\epsilon$ for every $\epsilon > 10^{-12}$.

**Part c)** Confirm the boundary identity. By definition the boundary of
$\Lambda_\epsilon$ is where $\|(zI - A)^{-1}\| = 1/\epsilon$ exactly. Find such
a point by bisection along the real direction from the first eigenvalue, for
$\epsilon = 10^{-1}$ and $10^{-2}$, and confirm the resolvent norm there equals
$1/\epsilon$ to a relative $10^{-6}$.

**Part d)** Confirm the third definition — that $z$ on the boundary really is
an eigenvalue of a nearby matrix — by *constructing* the perturbation. Let
$zI - A = U\Sigma V^{*}$ with smallest singular triple
$(\sigma_{\min}, \mathbf{u}, \mathbf{v})$, so $(zI-A)\mathbf{v} =
\sigma_{\min}\mathbf{u}$. Setting

$$
E = \sigma_{\min}\,\mathbf{u}\mathbf{v}^{*}
$$

gives $(zI - A - E)\mathbf{v} = \sigma_{\min}\mathbf{u} -
\sigma_{\min}\mathbf{u} = \mathbf{0}$, so $z$ is an eigenvalue of $A + E$
with eigenvector $\mathbf{v}$, and $\|E\|_2 = \sigma_{\min} = \epsilon$
because a rank-one matrix's only singular value is the product of the two unit
vectors' scale.

Build it — with `numpy`'s convention `U, s, Vh = svd(M)` the right singular
vector is `Vh[-1].conj()`, so $\mathbf{v}^{*}$ is `Vh[-1]` — and confirm
$\|E\|_2 = \epsilon$ to $10^{-12}$ and that $z$ is an eigenvalue of $A + E$
to $10^{-10}$. The sign and the conjugate both matter: getting either wrong
leaves $\|E\|$ correct while putting $z$ about $0.29$ away from the spectrum,
which is a failure the norm check alone would not catch.

**Part e)** Confirm the contrast with a normal matrix. For
$D = \operatorname{diag}(1, 2, 3)$ and 200 random $z$, confirm
$\sigma_{\min}(zI - D) = \min_i|z - \lambda_i|$ to $10^{-12}$ — so
$\Lambda_\epsilon(D)$ is exactly the union of $\epsilon$-discs, with no bulge
at all. For a normal matrix the pseudospectra carry no information the
eigenvalues did not already have, which is precisely why they are only
interesting in the non-normal case.

**Part f)** Draw the pseudospectral contours of the $32\times32$ Grcar matrix
with `la.pseudospectrum`, eigenvalues marked.

```{admonition} With your assistant
:class: tip
The transient of Exercise 5 and the pseudospectra of this one are connected by
the **Kreiss constant**, $\mathcal{K}(A) = \sup_{|z|>1}(|z| - 1)\,\|(zI -
A)^{-1}\|$, which satisfies $\mathcal{K}(A) \le \sup_k\|A^k\| \le
e\,n\,\mathcal{K}(A)$: pseudospectra bounding a transient from both sides. Ask
your assistant to write `kreiss_constant(A, n_grid)` estimating the supremum
over a polar grid outside the unit circle. Then check it against the
mathematics rather than against a picture: for the three matrices of Exercise 5
verify the lower bound $\mathcal{K}(A) \le \max_k\|A^k\|$ holds in all three
cases, that the upper bound $\max_k\|A^k\| \le e\,n\,\mathcal{K}(A)$ holds
with $n = 2$, and — the check that matters — that $\mathcal{K}$ *increases*
with $a$ in step with the peak, since the whole claim is that this quantity
sees what the spectrum cannot. The check is yours.
```

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

All three definitions in {eq}`eq-nn-pseudospectrum` are checked, and the third
is checked *constructively*: rather than asserting that a boundary point is an
eigenvalue of some nearby matrix, the exercise builds the rank-one perturbation
that makes it one and verifies both its norm and its effect.

In [ ]:
validate.check(
    defn_gap < 1e-10,
    "the resolvent and sigma_min definitions of the pseudospectrum agree (Eq. 6)",
    f"worst relative gap {defn_gap:.2e} over 200 random z: the SVD form is how "
    "it is computed, one decomposition per grid point",
)
validate.check(
    smin_at_eig < 1e-12,
    "and the spectrum lies inside every pseudospectrum",
    f"sigma_min(lambda I - A) <= {smin_at_eig:.2e} at each eigenvalue, so the "
    "resolvent norm is effectively infinite there",
)
validate.check(
    max(boundary_rel) < 1e-6,
    "on the boundary of Lambda_eps the resolvent norm equals 1/eps (Eq. 6)",
    f"worst relative deviation {max(boundary_rel):.2e} at eps = 1e-1 and 1e-2, "
    "located by bisection",
)
validate.check(
    abs(E_norm - eps_star) < 1e-12 and z_is_eig < 1e-10,
    "and a boundary z IS an eigenvalue of an explicit A + E with ||E|| = eps",
    f"the rank-one E built from the smallest singular triple has norm "
    f"{E_norm:.10f} and makes z an eigenvalue to {z_is_eig:.2e} — the third "
    "definition, constructed rather than asserted",
)
validate.check(
    normal_gap_ps < 1e-12,
    "while for a normal matrix the pseudospectra are exactly discs",
    f"sigma_min(zI - D) = min|z - lambda| to {normal_gap_ps:.2e} over 200 "
    "points, so they carry no information the eigenvalues did not already have",
)

---
## Notebook summary

**Similarity keeps the spectrum and little else.** Eigenvalues, trace and
determinant survived $X^{-1}AX$ to $10^{-15}$; the 2-norm did not, and a
general similarity applied to a symmetric matrix destroyed its symmetry
($\|M - M^{\top}\| \approx 1$) while leaving the eigenvalues untouched. An
*orthogonal* similarity kept symmetry, singular values and norm — which is
exactly why [§3.2](spectral-theorem.ipynb) required one.

**Schur always works.** $A = QTQ^{*}$ reconstructed both test matrices to
$10^{-15}$ with $Q$ unitary and $T$ exactly triangular, and
$\operatorname{cond}(Q) = 1$ in every case. The off-diagonal Frobenius mass of
$T$ matched the departure from normality
$\sqrt{\|A\|_F^2 - \sum|\lambda_i|^2}$ to $10^{-14}$ — a quantity defined
without reference to any decomposition, so the two could have disagreed. The
real Schur form came back quasi-triangular, with one $2\times2$ block per
complex-conjugate pair.

**The Jordan form dissolves at an exactly known rate.** SymPy produced
$PJP^{-1} = A$ exactly over the rationals. Numerically, putting $\delta$ in the
corner of $J_n(0)$ gave $n$ eigenvalues of modulus **exactly** $\delta^{1/n}$
— matched to a relative $10^{-15}$ — equally spaced by $2\pi/n$ to $10^{-15}$,
because the characteristic equation is $\lambda^n = \delta$. At $n = 6$,
$\delta = 10^{-14}$ that is a displacement of $4.6\times10^{-3}$, an
amplification of $4.6\times10^{11}$. And all 20 random perturbations of norm
$10^{-13}$ made $J_6(0)$ diagonalizable: defectiveness is not decidable in
`float64`, which is why `scipy` ships `schur` and not `jordan`.

**Bauer–Fike replaces perfect conditioning.** With $\operatorname{cond}(X) =
4.2\times10^{6}$ the bound held over 100 perturbations with a worst ratio of
$0.15$, while a symmetric matrix with the *same spectrum* under the *same*
perturbations obeyed Weyl with $\operatorname{cond}(X) = 1$. The non-normal
eigenvalues moved orders of magnitude further for identical $\|E\|$.

**A spectral radius of 0.8 is compatible with amplifying by six.**
$\left[\begin{smallmatrix}0.8&3\\0&0.8\end{smallmatrix}\right]$ has
$\rho = 0.8$, so $A^k \to 0$ is guaranteed — and it reaches
$\|A^4\|_2 = 6.17$ first, decaying to $4\times10^{-4}$ only by $k = 59$. The
normal matrix with the same spectrum satisfies $\|A^k\| = 0.8^k$ to $10^{-16}$
and can never rise. Varying the off-diagonal entry over $0.9, 1.5, 3$ changed
the peak by a factor of 3.2 while leaving the spectrum identical to $10^{-16}$.

**Pseudospectra see what eigenvalues cannot.** All three definitions agreed:
resolvent norm against $1/\sigma_{\min}$ to $10^{-13}$, the spectrum inside
every pseudospectrum, and the resolvent norm equal to $1/\epsilon$ on the
boundary to $10^{-15}$. The third definition was checked *constructively* — an
explicit rank-one $E$ with $\|E\|_2 = \epsilon$ making a boundary point a
genuine eigenvalue of $A + E$ to $10^{-14}$. For $\operatorname{diag}(1,2,3)$
the pseudospectra are exactly discs to $10^{-16}$, which is why they are only
interesting in the non-normal case.

**Methods introduced.** `scipy.linalg.schur` in both real and complex form,
the departure from normality, `sympy.Matrix.jordan_form`, lexicographic sorting
of complex spectra, the Bauer–Fike ratio, $\|A^k\|$ curves,
$\sigma_{\min}(zI - A)$ as the computable pseudospectrum, the rank-one
perturbation from the smallest singular triple, and `ecp.linalg.pseudospectrum`.

## Outlook

- **The algorithm behind Schur.** `schur` reduces to Hessenberg form and then
  runs shifted $QR$ iterations, and the whole of that is
  [§5.2](../05-numerical/eigenvalue-algorithms.ipynb) — including why the
  shifts matter and why nobody forms a characteristic polynomial.
- **Non-normality in iterative solvers.** GMRES convergence is governed by
  pseudospectra rather than eigenvalues for exactly the reasons here, which is
  why a preconditioner that clusters the spectrum may not help a non-normal
  system. [§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb) measures
  it.
- **Where the transient is the physics.** Fluid flows that are linearly stable
  by every eigenvalue can still amplify a disturbance by $10^{3}$ and become
  turbulent, and this notebook's $6\times$ is a toy version of that. Trefethen
  and Embree {cite}`trefethen2005spectra` treat the real case.
- **Singular values instead.** Every difficulty here came from eigenvalues of a
  non-normal matrix. Singular values have none of these problems: they are
  always real and non-negative, always perfectly conditioned, and always come
  with orthonormal bases on both sides.
  [§4.1](../04-svd/svd-geometry.ipynb) opens the sustained argument for using
  them instead.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()